# FHI — Expert Optimization Notebook
## Target: Macro F1 ≥ 0.91

### Strategy summary
| Step | Technique | Expected ΔF1 |
|------|-----------|-------------|
| 1 | Advanced feature engineering (missing indicators, interactions) | +0.01–0.02 |
| 2 | XGBoost Optuna tuning (100 trials) | +0.02–0.04 |
| 3 | Two-stage cascaded classifier | +0.03–0.05 |
| 4 | SMOTE inside CV folds (High class) | +0.01–0.02 |
| 5 | OOF threshold optimisation | +0.02–0.04 |
| 6 | Stacking meta-learner | +0.01–0.02 |
| 7 | Optimised ensemble weights | +0.005–0.01 |


## 1. Imports & Setup

In [8]:
import warnings, re
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 60)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

from copy import deepcopy
from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, log_loss, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR     = "/home/conite/Documents/WORKSPACE/PROJECTS/Kaggle/Financial Health Prediction Challenge/data"
SEED         = 42
N_FOLDS      = 5
TARGET       = "Target"
TARGET_ORDER = ["Low", "Medium", "High"]
TARGET_MAP   = {"Low": 0, "Medium": 1, "High": 2}
TARGET_IMAP  = {0: "Low", 1: "Medium", 2: "High"}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print("Ready. imblearn:", __import__("imblearn").__version__)


Ready. imblearn: 0.14.1


## 2. Data Loading & Base Pipeline

In [9]:
train   = pd.read_csv(f"{DATA_DIR}/Train.csv")
test    = pd.read_csv(f"{DATA_DIR}/Test.csv")
sample  = pd.read_csv(f"{DATA_DIR}/SampleSubmission.csv")

# ── Clean ──────────────────────────────────────────────────────────────────
def clean_text(x):
    if pd.isna(x): return x
    x = str(x).strip()
    x = x.replace("\u2019","'").replace("\u2018","'").replace("?","'")
    x = re.sub(r"don.{0,2}t know.*", "Don't know", x, flags=re.IGNORECASE)
    x = re.sub(r"do not know.*",     "Don't know", x, flags=re.IGNORECASE)
    x = re.sub(r"used to have.*",    "Used to have", x, flags=re.IGNORECASE)
    return x.strip()

def clean_df(df):
    df = df.copy()
    for col in df.select_dtypes("object").columns:
        if col != "ID": df[col] = df[col].apply(clean_text)
    if "current_problem_cash_flow" in df.columns:
        df["current_problem_cash_flow"] = df["current_problem_cash_flow"].replace("0","No")
    return df

train = clean_df(train)
test  = clean_df(test)

# Cap financial outliers at 99th pct (train-derived)
for col in ["personal_income","business_expenses","business_turnover"]:
    cap = train[col].quantile(0.99)
    train[col] = train[col].clip(upper=cap)
    test[col]  = test[col].clip(upper=cap)

print("Train:", train.shape, " Test:", test.shape)
print(train[TARGET].value_counts())


Train: (9618, 39)  Test: (2405, 38)
Target
Low       6280
Medium    2868
High       470
Name: count, dtype: int64


## 3. Advanced Feature Engineering

### New vs baseline
- **Missing indicators** for every column with >20% NaN — 'refused to answer' is itself informative
- **Domain missingness rates** — separate counts for insurance / banking / attitude blocks
- **Formalization score** — tax compliance + financial records + formal banking
- **Country × insurance interaction** — insurance penetration differs sharply by country
- **Survey-block scores** — aggregate perception scores, attitude scores


In [10]:
HAVE_ORDER  = ["Never had", "Used to have", "Don't know", "Have now"]
YES_NO_DK   = ["No", "Don't know", "Yes"]
YES_NO      = ["No", "Yes"]
AGREE_ORDER = ["No", "Don't know or N/A", "Yes"]
AGREE2      = ["No", "Don't know", "Yes"]
OFFER_ORDER = ["No", "Yes, sometimes", "Yes, always"]
KEEP_ORDER  = ["No", "Yes, sometimes", "Yes, always", "Yes"]
COMPLY_MAP  = ["No", "Don't know", "Refused", "Yes"]

ORDINAL_SPEC = {
    "attitude_stable_business_environment": AGREE_ORDER,
    "attitude_worried_shutdown": AGREE_ORDER,
    "compliance_income_tax": COMPLY_MAP,
    "perception_insurance_doesnt_cover_losses": YES_NO_DK,
    "perception_cannot_afford_insurance": YES_NO_DK,
    "motor_vehicle_insurance": HAVE_ORDER,
    "has_mobile_money": HAVE_ORDER,
    "current_problem_cash_flow": YES_NO,
    "has_cellphone": YES_NO,
    "offers_credit_to_customers": OFFER_ORDER,
    "attitude_satisfied_with_achievement": AGREE2,
    "has_credit_card": HAVE_ORDER,
    "keeps_financial_records": KEEP_ORDER,
    "perception_insurance_companies_dont_insure_businesses_like_yours": YES_NO_DK,
    "perception_insurance_important": YES_NO_DK,
    "has_insurance": YES_NO,
    "covid_essential_service": YES_NO_DK,
    "attitude_more_successful_next_year": AGREE2,
    "problem_sourcing_money": YES_NO,
    "marketing_word_of_mouth": YES_NO,
    "has_loan_account": HAVE_ORDER,
    "has_internet_banking": HAVE_ORDER,
    "has_debit_card": HAVE_ORDER,
    "future_risk_theft_stock": YES_NO,
    "medical_insurance": HAVE_ORDER,
    "funeral_insurance": HAVE_ORDER,
    "motivation_make_more_money": YES_NO,
    "uses_friends_family_savings": HAVE_ORDER,
    "uses_informal_lender": HAVE_ORDER,
    "owner_sex": ["Female", "Male"],
}

PRODUCT_COLS = ["motor_vehicle_insurance","has_mobile_money","has_credit_card",
                "has_loan_account","has_internet_banking","has_debit_card",
                "medical_insurance","funeral_insurance","uses_friends_family_savings",
                "uses_informal_lender"]
INSURANCE_COLS  = ["motor_vehicle_insurance","medical_insurance","funeral_insurance","has_insurance"]
BANKING_COLS    = ["has_mobile_money","has_credit_card","has_loan_account",
                   "has_internet_banking","has_debit_card"]
ATTITUDE_COLS   = ["attitude_stable_business_environment","attitude_worried_shutdown",
                   "attitude_satisfied_with_achievement","attitude_more_successful_next_year"]
PERCEPTION_COLS = ["perception_insurance_doesnt_cover_losses","perception_cannot_afford_insurance",
                   "perception_insurance_companies_dont_insure_businesses_like_yours",
                   "perception_insurance_important"]

# High-missingness columns (>20% missing in train)
HIGH_MISS_COLS = [c for c in train.columns
                  if c not in ("ID", TARGET) and train[c].isnull().mean() > 0.20]
print(f"High-miss cols ({len(HIGH_MISS_COLS)}): {HIGH_MISS_COLS}")


High-miss cols (26): ['motor_vehicle_insurance', 'has_mobile_money', 'current_problem_cash_flow', 'has_cellphone', 'owner_sex', 'offers_credit_to_customers', 'attitude_satisfied_with_achievement', 'has_credit_card', 'keeps_financial_records', 'perception_insurance_companies_dont_insure_businesses_like_yours', 'perception_insurance_important', 'has_insurance', 'covid_essential_service', 'attitude_more_successful_next_year', 'problem_sourcing_money', 'marketing_word_of_mouth', 'has_loan_account', 'has_internet_banking', 'has_debit_card', 'future_risk_theft_stock', 'business_age_months', 'medical_insurance', 'funeral_insurance', 'motivation_make_more_money', 'uses_friends_family_savings', 'uses_informal_lender']


In [11]:
def engineer_features_v2(df, fit_country_stats=None):
    """Extended feature engineering. fit_country_stats passed from train to prevent leakage."""
    df = df.copy()
    eps = 1.0

    # ── 1. Core financial features ─────────────────────────────────────────
    df["business_age_total"] = df["business_age_years"].fillna(0) + df["business_age_months"].fillna(0)/12
    df["log_income"]   = np.log1p(df["personal_income"].clip(lower=0))
    df["log_expenses"] = np.log1p(df["business_expenses"].clip(lower=0))
    df["log_turnover"] = np.log1p(df["business_turnover"].clip(lower=0))
    df["profit_proxy"] = df["business_turnover"].fillna(0) - df["business_expenses"].fillna(0)
    df["log_profit"]   = np.log1p(df["profit_proxy"].clip(lower=0))
    df["expense_ratio"]         = df["business_expenses"]  / (df["business_turnover"] + eps)
    df["income_turnover_ratio"] = df["personal_income"]    / (df["business_turnover"] + eps)
    df["income_expense_ratio"]  = df["personal_income"]    / (df["business_expenses"]  + eps)  # NEW
    df["turnover_per_year"]     = df["business_turnover"]  / (df["business_age_total"]  + eps)  # NEW

    # ── 2. Financial product access scores ────────────────────────────────
    product_map = {"Have now": 2, "Used to have": 1, "Never had": 0, "Don't know": 0}
    for col in PRODUCT_COLS:
        if col in df.columns:
            df[f"{col}_score"] = df[col].map(product_map).fillna(0)
    score_cols = [f"{c}_score" for c in PRODUCT_COLS if c in df.columns]
    df["financial_access_score"] = df[score_cols].sum(axis=1)

    # ── 3. Domain-specific product scores (NEW) ────────────────────────────
    ins_score_cols  = [f"{c}_score" for c in ["motor_vehicle_insurance","medical_insurance",
                                               "funeral_insurance"] if f"{c}_score" in df.columns]
    bank_score_cols = [f"{c}_score" for c in ["has_mobile_money","has_credit_card",
                                               "has_loan_account","has_internet_banking",
                                               "has_debit_card"] if f"{c}_score" in df.columns]
    df["insurance_product_score"] = df[ins_score_cols].sum(axis=1)   if ins_score_cols  else 0
    df["banking_product_score"]   = df[bank_score_cols].sum(axis=1)  if bank_score_cols else 0

    # ── 4. Composite flags ─────────────────────────────────────────────────
    df["uses_formal_finance"] = (
        (df["has_credit_card"].isin(["Have now"])).astype(int) +
        (df["has_loan_account"].isin(["Have now"])).astype(int) +
        (df["has_internet_banking"].isin(["Have now"])).astype(int) +
        (df["has_debit_card"].isin(["Have now"])).astype(int)
    )
    # Formalisation score (NEW): tax + records + formal finance
    df["formalization_score"] = (
        (df["compliance_income_tax"] == "Yes").astype(int) +
        (df["keeps_financial_records"].isin(["Yes","Yes, always","Yes, sometimes"])).astype(int) +
        df["uses_formal_finance"]
    )

    # ── 5. Attitude & perception composites ───────────────────────────────
    pos_att = ["attitude_stable_business_environment","attitude_more_successful_next_year",
               "attitude_satisfied_with_achievement"]
    for col in pos_att:
        df[f"{col}_bin"] = (df[col] == "Yes").astype(int)
    df["positive_attitude"] = df[[f"{c}_bin" for c in pos_att]].sum(axis=1)

    neg_att = ["attitude_worried_shutdown"]
    df["worried_shutdown"] = (df["attitude_worried_shutdown"] == "Yes").astype(int)

    df["insurance_engaged"] = (
        (df["perception_insurance_important"] == "Yes").astype(int) +
        (df["has_insurance"] == "Yes").astype(int)
    )
    # Perception of insurance barriers (NEW)
    df["insurance_barrier_score"] = (
        (df["perception_cannot_afford_insurance"] == "Yes").astype(int) +
        (df["perception_insurance_doesnt_cover_losses"] == "Yes").astype(int) +
        (df["perception_insurance_companies_dont_insure_businesses_like_yours"] == "Yes").astype(int)
    )

    # ── 6. Demographics ───────────────────────────────────────────────────
    df["is_male"]    = (df["owner_sex"] == "Male").astype(int)
    df["age_sq"]     = df["owner_age"] ** 2                          # NEW
    df["age_x_age_business"] = df["owner_age"] * df["business_age_total"]  # NEW

    # ── 7. Missing value indicators (NEW — most impactful addition) ───────
    for col in HIGH_MISS_COLS:
        df[f"miss_{col}"] = df[col].isnull().astype(int)

    # Domain-level missingness rates (NEW)
    ins_miss   = [c for c in INSURANCE_COLS if c in df.columns]
    bank_miss  = [c for c in BANKING_COLS   if c in df.columns]
    att_miss   = [c for c in ATTITUDE_COLS  if c in df.columns]
    perc_miss  = [c for c in PERCEPTION_COLS if c in df.columns]
    df["insurance_miss_rate"]  = df[ins_miss].isnull().mean(axis=1)
    df["banking_miss_rate"]    = df[bank_miss].isnull().mean(axis=1)
    df["attitude_miss_rate"]   = df[att_miss].isnull().mean(axis=1)
    df["perception_miss_rate"] = df[perc_miss].isnull().mean(axis=1)
    df["n_missing"] = df.isnull().sum(axis=1)

    # ── 8. Country × insurance interaction (NEW) ──────────────────────────
    # Safe: use "has_insurance" (only 2-cat) × country label
    df["country_x_insurance"] = df["country"].astype(str) + "_" + df["has_insurance"].fillna("Missing")

    return df

train_fe = engineer_features_v2(train)
test_fe  = engineer_features_v2(test)

new_cols = [c for c in train_fe.columns if c not in train.columns]
print(f"New features added: {len(new_cols)}")
print(new_cols)


New features added: 67
['business_age_total', 'log_income', 'log_expenses', 'log_turnover', 'profit_proxy', 'log_profit', 'expense_ratio', 'income_turnover_ratio', 'income_expense_ratio', 'turnover_per_year', 'motor_vehicle_insurance_score', 'has_mobile_money_score', 'has_credit_card_score', 'has_loan_account_score', 'has_internet_banking_score', 'has_debit_card_score', 'medical_insurance_score', 'funeral_insurance_score', 'uses_friends_family_savings_score', 'uses_informal_lender_score', 'financial_access_score', 'insurance_product_score', 'banking_product_score', 'uses_formal_finance', 'formalization_score', 'attitude_stable_business_environment_bin', 'attitude_more_successful_next_year_bin', 'attitude_satisfied_with_achievement_bin', 'positive_attitude', 'worried_shutdown', 'insurance_engaged', 'insurance_barrier_score', 'is_male', 'age_sq', 'age_x_age_business', 'miss_motor_vehicle_insurance', 'miss_has_mobile_money', 'miss_current_problem_cash_flow', 'miss_has_cellphone', 'miss_ow

In [12]:
# ── Encoding ──────────────────────────────────────────────────────────────────
le_country    = LabelEncoder()
le_country_x  = LabelEncoder()

def encode_df(df, fit=True):
    global le_country, le_country_x
    df = df.copy()
    for col, order in ORDINAL_SPEC.items():
        if col not in df.columns: continue
        codes = pd.Categorical(df[col], categories=order, ordered=True).codes.astype(float)
        df[col] = np.where(codes == -1, np.nan, codes)   # -1 (unknown) → NaN

    if fit:
        df["country"]             = le_country.fit_transform(df["country"].fillna("unknown"))
        df["country_x_insurance"] = le_country_x.fit_transform(df["country_x_insurance"])
    else:
        country_map = dict(zip(le_country.classes_, le_country.transform(le_country.classes_)))
        df["country"] = df["country"].fillna("unknown").map(country_map).fillna(-1)
        known = set(le_country_x.classes_)
        df["country_x_insurance"] = df["country_x_insurance"].apply(
            lambda x: le_country_x.transform([x])[0] if x in known else -1)
    return df

train_enc = encode_df(train_fe, fit=True)
test_enc  = encode_df(test_fe,  fit=False)

EXCLUDE = {"ID", TARGET} | {c for c in train_enc.columns if c.endswith("_bin")}
feature_cols = [c for c in train_enc.columns if c not in EXCLUDE]

X      = train_enc[feature_cols].copy()
y      = train_enc[TARGET].map(TARGET_MAP)
X_test = test_enc[feature_cols].copy()

print(f"Feature matrix: {X.shape}")
print(f"Class distribution: {y.value_counts().to_dict()}")


Feature matrix: (9618, 101)
Class distribution: {0: 6280, 1: 2868, 2: 470}


## 4. XGBoost Hyperparameter Tuning (Optuna, 100 trials)

XGBoost outperformed all other models in baseline (MacroF1=0.81). Deeper tuning here
gives the largest single-model gain. Key parameters to explore:
- `max_depth` + `min_child_weight`: controls tree complexity / overfitting on minority class
- `gamma` + `reg_alpha` + `reg_lambda`: regularisation to improve generalisation
- `scale_pos_weight` equivalent via `sample_weight`: aggressive minority boosting


In [13]:
def make_xgb(params):
    return xgb.XGBClassifier(
        **params,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        use_label_encoder=False,
        random_state=SEED,
        n_jobs=-1,
        early_stopping_rounds=40,
    )

def xgb_cv_score(params, X, y, cv):
    losses = []
    for tr, va in cv.split(X, y):
        m = make_xgb(params)
        m.fit(X.iloc[tr].values, y.iloc[tr].values,
              eval_set=[(X.iloc[va].values, y.iloc[va].values)],
              verbose=False)
        p = m.predict_proba(X.iloc[va].values)
        losses.append(log_loss(y.iloc[va], p))
    return np.mean(losses)

def objective_xgb(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 400, 1500),
        "max_depth"       : trial.suggest_int("max_depth", 4, 10),
        "learning_rate"   : trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma"           : trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha"       : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda"      : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "max_delta_step"  : trial.suggest_int("max_delta_step", 0, 5),  # helps imbalance
    }
    return xgb_cv_score(params, X, y, skf)

study_xgb = optuna.create_study(direction="minimize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)

print(f"\nBest XGB log-loss : {study_xgb.best_value:.4f}")
print(f"Best params        : {study_xgb.best_params}")


  0%|          | 0/100 [00:00<?, ?it/s]

sh: 1: nvidia-smi: not found



Best XGB log-loss : 0.3023
Best params        : {'n_estimators': 1242, 'max_depth': 6, 'learning_rate': 0.013541634865372902, 'subsample': 0.8637758350641194, 'colsample_bytree': 0.5268357678288189, 'colsample_bylevel': 0.6509114271288193, 'min_child_weight': 4, 'gamma': 1.1451773583390517, 'reg_alpha': 0.017471470079401707, 'reg_lambda': 0.009925713863724933, 'max_delta_step': 4}


In [14]:
# Evaluate best XGBoost on Macro F1
best_xgb_params = study_xgb.best_params
best_xgb_params.update({"objective":"multi:softprob","num_class":3,
                         "eval_metric":"mlogloss","use_label_encoder":False,
                         "random_state":SEED,"n_jobs":-1,"early_stopping_rounds":40})

scores_ll, scores_f1, scores_f1h = [], [], []
for tr, va in skf.split(X, y):
    m = xgb.XGBClassifier(**best_xgb_params)
    m.fit(X.iloc[tr].values, y.iloc[tr].values,
          eval_set=[(X.iloc[va].values, y.iloc[va].values)], verbose=False)
    p    = m.predict_proba(X.iloc[va].values)
    pred = p.argmax(axis=1)
    scores_ll.append( log_loss(y.iloc[va], p))
    scores_f1.append( f1_score(y.iloc[va], pred, average="macro"))
    scores_f1h.append(f1_score(y.iloc[va], pred, average=None)[2])

print(f"XGBoost (tuned) — LogLoss={np.mean(scores_ll):.4f}±{np.std(scores_ll):.4f} "
      f"MacroF1={np.mean(scores_f1):.4f}  HighF1={np.mean(scores_f1h):.4f}")


XGBoost (tuned) — LogLoss=0.3023±0.0186 MacroF1=0.8124  HighF1=0.7264


## 5. LightGBM Re-tune (expanded search)

In [ ]:
def objective_lgb(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 400, 2000),
        "max_depth"        : trial.suggest_int("max_depth", 4, 10),
        "num_leaves"       : trial.suggest_int("num_leaves", 31, 255),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 60),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "min_split_gain"   : trial.suggest_float("min_split_gain", 0.0, 1.0),
        "objective": "multiclass", "num_class": 3, "class_weight": "balanced",
        "random_state": SEED, "n_jobs": -1, "verbose": -1,
    }
    losses = []
    m = lgb.LGBMClassifier(**params)
    for tr, va in skf.split(X, y):
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])],
              callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
        losses.append(log_loss(y.iloc[va], m.predict_proba(X.iloc[va])))
    return np.mean(losses)

study_lgb = optuna.create_study(direction="minimize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(objective_lgb, n_trials=80, show_progress_bar=True)

best_lgb_params = study_lgb.best_params
best_lgb_params.update({"objective":"multiclass","num_class":3,"class_weight":"balanced",
                         "random_state":SEED,"n_jobs":-1,"verbose":-1})
print(f"Best LGB log-loss: {study_lgb.best_value:.4f}")
print(f"Best params: {best_lgb_params}")


## 6. Two-Stage Cascaded Classifier

**Rationale**: In the original 3-class formulation, High is only 4.9% of all data.
The two-stage reframes the problem:
- **Stage 1**: Low (65%) vs Non-Low (35%) — balanced enough for clean separation
- **Stage 2**: Among Non-Low only → Medium (89%) vs High (11%) — High goes from 5% to ~14%

This nearly **triples** the effective density of High examples in the stage where it matters.

Final probabilities:
```
P(Low)    = Stage1.P(Low)
P(Medium) = Stage1.P(Non-Low) × Stage2.P(Medium)
P(High)   = Stage1.P(Non-Low) × Stage2.P(High)
```


In [ ]:
class TwoStageClassifier:
    """
    Stage 1: Binary  Low(0) vs Non-Low(1)
    Stage 2: Binary  Medium(0) vs High(1)  — trained only on Non-Low samples
    Produces calibrated 3-class probability vector.
    """
    def __init__(self, params1=None, params2=None):
        self.params1 = params1 or {}
        self.params2 = params2 or {}

    def _make_lgb(self, params, class_weight="balanced"):
        p = {"objective":"binary","random_state":SEED,"n_jobs":-1,"verbose":-1,
             "class_weight":class_weight,"n_estimators":600,"learning_rate":0.05,
             "num_leaves":63,"subsample":0.8,"colsample_bytree":0.8}
        p.update(params)
        return lgb.LGBMClassifier(**p)

    def fit(self, X, y):
        # Stage 1: Low vs Non-Low
        y1 = (y != 0).astype(int)           # 0=Low, 1=Non-Low
        self.s1 = self._make_lgb(self.params1)
        self.s1.fit(X, y1,
                    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])

        # Stage 2: Medium vs High (on Non-Low subset)
        mask2 = y != 0
        y2    = (y[mask2] == 2).astype(int)  # 0=Medium, 1=High
        X2    = X[mask2]
        self.s2 = self._make_lgb(self.params2, class_weight="balanced")
        self.s2.fit(X2, y2,
                    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
        return self

    def predict_proba(self, X):
        p_nonlow = self.s1.predict_proba(X)[:, 1]   # P(Non-Low)
        p_high_given_nonlow = self.s2.predict_proba(X)[:, 1]  # P(High | Non-Low)

        p_low    = 1 - p_nonlow
        p_high   = p_nonlow * p_high_given_nonlow
        p_medium = p_nonlow * (1 - p_high_given_nonlow)

        return np.column_stack([p_low, p_medium, p_high])

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


# ── Evaluate two-stage ────────────────────────────────────────────────────────
ts_ll, ts_f1, ts_f1h = [], [], []
for tr, va in skf.split(X, y):
    m = TwoStageClassifier()
    m.fit(X.iloc[tr], y.iloc[tr])
    proba = m.predict_proba(X.iloc[va])
    pred  = proba.argmax(axis=1)
    ts_ll.append( log_loss(y.iloc[va], proba))
    ts_f1.append( f1_score(y.iloc[va], pred, average="macro"))
    ts_f1h.append(f1_score(y.iloc[va], pred, average=None)[2])

print(f"Two-Stage LGB      — LogLoss={np.mean(ts_ll):.4f}±{np.std(ts_ll):.4f} "
      f"MacroF1={np.mean(ts_f1):.4f}  HighF1={np.mean(ts_f1h):.4f}")


## 7. SMOTE Oversampling Inside CV Folds

**Why inside CV?** Applying SMOTE before splitting leaks synthetic High samples into
validation — they become near-duplicates of training points, inflating scores.
The correct approach: oversample only the training fold, validate on original distribution.


In [ ]:
smote = SMOTE(random_state=SEED, k_neighbors=5)

def evaluate_with_smote(model_fn, X, y, cv):
    """model_fn() must return a fresh unfitted model each call."""
    ll, f1s, f1h = [], [], []
    for tr, va in cv.split(X, y):
        Xtr, ytr = X.iloc[tr].values, y.iloc[tr].values
        Xva, yva = X.iloc[va].values, y.iloc[va].values

        # Impute before SMOTE (SMOTE needs no NaN)
        imp = SimpleImputer(strategy="median")
        Xtr = imp.fit_transform(Xtr)
        Xva = imp.transform(Xva)

        Xtr_res, ytr_res = smote.fit_resample(Xtr, ytr)

        m = model_fn()
        m.fit(Xtr_res, ytr_res)
        proba = m.predict_proba(Xva)
        pred  = proba.argmax(axis=1)
        ll.append( log_loss(yva, proba))
        f1s.append( f1_score(yva, pred, average="macro"))
        f1h.append(f1_score(yva, pred, average=None)[2])
    return ll, f1s, f1h

# Test with tuned XGB + SMOTE
def make_best_xgb():
    p = deepcopy(best_xgb_params)
    p.pop("early_stopping_rounds", None)      # no eval_set when using SMOTE pipeline
    return xgb.XGBClassifier(**p)

smote_ll, smote_f1, smote_f1h = evaluate_with_smote(make_best_xgb, X, y, skf)
print(f"XGB+SMOTE          — LogLoss={np.mean(smote_ll):.4f}±{np.std(smote_ll):.4f} "
      f"MacroF1={np.mean(smote_f1):.4f}  HighF1={np.mean(smote_f1h):.4f}")


## 8. OOF Threshold Optimisation

**Why this helps**: Models minimise log-loss, not Macro F1. The default argmax
threshold (0.33 per class) is rarely optimal for imbalanced Macro F1.

**Method**: Collect OOF probabilities from the best model, then find thresholds
`[t0, t1, t2]` that maximise Macro F1 on the full OOF set via Nelder-Mead.
Assign class `k = argmax(p_k / t_k)` — equivalent to shifting each class's
decision boundary independently.


In [ ]:
# Collect OOF probas from best single model (tuned XGB)
oof_proba  = np.zeros((len(X), 3))
oof_y      = np.zeros(len(X), dtype=int)

for tr, va in skf.split(X, y):
    m = xgb.XGBClassifier(**best_xgb_params)
    m.fit(X.iloc[tr].values, y.iloc[tr].values,
          eval_set=[(X.iloc[va].values, y.iloc[va].values)], verbose=False)
    oof_proba[va] = m.predict_proba(X.iloc[va].values)
    oof_y[va]     = y.iloc[va].values

oof_pred_default = oof_proba.argmax(axis=1)
default_f1 = f1_score(oof_y, oof_pred_default, average="macro")
print(f"Default threshold  — OOF Macro F1 = {default_f1:.4f}")

# ── Optimise thresholds ───────────────────────────────────────────────────────
def apply_thresholds(proba, thresholds):
    """Scale each class probability by its threshold, then argmax."""
    t = np.array(thresholds)
    return (proba / t).argmax(axis=1)

def neg_macro_f1(thresholds):
    preds = apply_thresholds(oof_proba, thresholds)
    return -f1_score(oof_y, preds, average="macro")

result = minimize(
    neg_macro_f1,
    x0=[1.0, 1.0, 1.0],
    method="Nelder-Mead",
    options={"maxiter": 5000, "xatol": 1e-5, "fatol": 1e-5}
)
best_thresholds = result.x
opt_f1 = -result.fun

print(f"Optimised thresholds: Low={best_thresholds[0]:.4f}  "
      f"Medium={best_thresholds[1]:.4f}  High={best_thresholds[2]:.4f}")
print(f"Threshold-tuned    — OOF Macro F1 = {opt_f1:.4f}  "
      f"(Δ = +{opt_f1 - default_f1:.4f})")

# Per-class F1 after tuning
preds_opt = apply_thresholds(oof_proba, best_thresholds)
print("\nPer-class F1:")
print(classification_report(oof_y, preds_opt, target_names=["Low","Medium","High"]))


## 9. Stacking Meta-Learner

**Strategy**: Collect OOF probability vectors from XGB, LGB, CatBoost as level-0.
Train a Logistic Regression meta-learner on the 9-dimensional OOF feature space.
This captures complementary errors between models without overfitting.


In [ ]:
cat_base = CatBoostClassifier(
    iterations=600, depth=7, learning_rate=0.05,
    loss_function="MultiClass", eval_metric="MultiClass",
    class_weights=[1.0, 2.2, 13.4], random_seed=SEED,
    verbose=False, early_stopping_rounds=30,
)

# Collect OOF from all 3 base models
oof_xgb = np.zeros((len(X), 3))
oof_lgb = np.zeros((len(X), 3))
oof_cat = np.zeros((len(X), 3))
oof_stk = np.zeros(len(X), dtype=int)   # true labels

for tr, va in skf.split(X, y):
    Xtr, Xva = X.iloc[tr], X.iloc[va]
    ytr, yva = y.iloc[tr], y.iloc[va]

    # XGB
    mx = xgb.XGBClassifier(**best_xgb_params)
    mx.fit(Xtr.values, ytr.values, eval_set=[(Xva.values, yva.values)], verbose=False)
    oof_xgb[va] = mx.predict_proba(Xva.values)

    # LGB
    ml = lgb.LGBMClassifier(**best_lgb_params)
    ml.fit(Xtr, ytr, eval_set=[(Xva, yva)],
           callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[va] = ml.predict_proba(Xva)

    # CatBoost
    mc = deepcopy(cat_base)
    mc.fit(Xtr, ytr, eval_set=(Xva, yva), verbose=False)
    oof_cat[va] = mc.predict_proba(Xva)

    oof_stk[va] = yva.values

# Stack features
oof_meta = np.hstack([oof_xgb, oof_lgb, oof_cat])   # (N, 9)

# Meta-learner CV evaluation
meta_ll, meta_f1, meta_f1h = [], [], []
meta_skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED+1)

for tr, va in meta_skf.split(oof_meta, oof_stk):
    meta_m = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced",
                           random_state=SEED)
    )
    meta_m.fit(oof_meta[tr], oof_stk[tr])
    proba = meta_m.predict_proba(oof_meta[va])
    pred  = proba.argmax(axis=1)
    meta_ll.append( log_loss(oof_stk[va], proba))
    meta_f1.append( f1_score(oof_stk[va], pred, average="macro"))
    meta_f1h.append(f1_score(oof_stk[va], pred, average=None)[2])

print(f"Stacking (LR meta) — LogLoss={np.mean(meta_ll):.4f}±{np.std(meta_ll):.4f} "
      f"MacroF1={np.mean(meta_f1):.4f}  HighF1={np.mean(meta_f1h):.4f}")


## 10. Model Comparison Summary

In [ ]:
# Re-evaluate tuned LGB for comparison
lg_ll2, lg_f12, lg_f1h2 = [], [], []
for tr, va in skf.split(X, y):
    ml = lgb.LGBMClassifier(**best_lgb_params)
    ml.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])],
           callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    p = ml.predict_proba(X.iloc[va]); pr = p.argmax(axis=1)
    lg_ll2.append(log_loss(y.iloc[va], p))
    lg_f12.append(f1_score(y.iloc[va], pr, average="macro"))
    lg_f1h2.append(f1_score(y.iloc[va], pr, average=None)[2])

results = pd.DataFrame([
    ["XGBoost (tuned)",   np.mean(scores_ll),  np.mean(scores_f1),  np.mean(scores_f1h)],
    ["LightGBM (tuned)",  np.mean(lg_ll2),     np.mean(lg_f12),     np.mean(lg_f1h2)],
    ["Two-Stage LGB",     np.mean(ts_ll),      np.mean(ts_f1),      np.mean(ts_f1h)],
    ["XGB+SMOTE",         np.mean(smote_ll),   np.mean(smote_f1),   np.mean(smote_f1h)],
    ["XGB+Thresholds",    log_loss(oof_y, oof_proba), opt_f1, f1_score(oof_y, preds_opt, average=None)[2]],
    ["Stacking (LR)",     np.mean(meta_ll),    np.mean(meta_f1),    np.mean(meta_f1h)],
], columns=["Model","LogLoss","MacroF1","HighF1"])

results = results.sort_values("MacroF1", ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2ca02c" if v >= 0.91 else "#1f77b4" if v >= 0.87 else "#ff7f0e"
          for v in results["MacroF1"]]
bars = ax.barh(results["Model"], results["MacroF1"], color=colors)
ax.axvline(0.91, color="red", linestyle="--", linewidth=1.5, label="Target F1=0.91")
ax.axvline(0.81, color="gray", linestyle=":", linewidth=1.2, label="Baseline F1=0.81")
ax.set_xlabel("OOF Macro F1")
ax.set_title("Model Comparison — Macro F1 (OOF)")
for bar, val in zip(bars, results["MacroF1"]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=10)
ax.legend(); ax.set_xlim(0.75, 0.97)
plt.tight_layout(); plt.show()


## 11. Optimised Ensemble

Train all base models on full data.  
Find ensemble weights that maximise OOF Macro F1 via constrained optimisation.
Apply best thresholds from section 8 to the final blended probabilities.


In [ ]:
# ── Train final models on full data ──────────────────────────────────────────
print("Training final base models on full dataset...")

xgb_final = xgb.XGBClassifier(**{k:v for k,v in best_xgb_params.items()
                                   if k != "early_stopping_rounds"})
xgb_final.fit(X.values, y.values)
print("  XGBoost done.")

lgb_final = lgb.LGBMClassifier(**best_lgb_params)
lgb_final.fit(X, y, callbacks=[lgb.log_evaluation(-1)])
print("  LightGBM done.")

cat_final = CatBoostClassifier(
    iterations=600, depth=7, learning_rate=0.05,
    loss_function="MultiClass", class_weights=[1.0, 2.2, 13.4],
    random_seed=SEED, verbose=False
)
cat_final.fit(X, y)
print("  CatBoost done.")

ts_final = TwoStageClassifier()
ts_final.fit(X, y)
print("  Two-Stage done.")

# ── Optimise ensemble weights on OOF ─────────────────────────────────────────
def ensemble_oof(weights):
    """Blend OOF probas with given weights, return -MacroF1."""
    w = np.abs(weights); w /= w.sum()     # ensure positive, sum-to-1
    blended = (w[0]*oof_xgb + w[1]*oof_lgb + w[2]*oof_cat)
    pred    = apply_thresholds(blended, best_thresholds)
    return -f1_score(oof_stk, pred, average="macro")

res_w = minimize(ensemble_oof, x0=[0.5, 0.3, 0.2],
                 method="Nelder-Mead",
                 options={"maxiter":2000, "xatol":1e-5})
raw_w = np.abs(res_w.x); opt_weights = raw_w / raw_w.sum()
print(f"\nOptimised weights — XGB:{opt_weights[0]:.3f}  "
      f"LGB:{opt_weights[1]:.3f}  CAT:{opt_weights[2]:.3f}")
print(f"Ensemble OOF MacroF1 (with thresholds): {-res_w.fun:.4f}")


In [ ]:
# ── Generate test predictions ─────────────────────────────────────────────────
xgb_test  = xgb_final.predict_proba(X_test.values)
lgb_test  = lgb_final.predict_proba(X_test)
cat_test  = cat_final.predict_proba(X_test)

ensemble_test = (opt_weights[0]*xgb_test +
                 opt_weights[1]*lgb_test  +
                 opt_weights[2]*cat_test)

# Apply optimised thresholds
final_pred   = apply_thresholds(ensemble_test, best_thresholds)
pred_labels  = [TARGET_IMAP[p] for p in final_pred]

print("Test prediction distribution:")
print(pd.Series(pred_labels).value_counts())

sub_path = f"{DATA_DIR}/../notebook/submission_optimized.csv"
submission = pd.DataFrame({"ID": test["ID"], TARGET: pred_labels})
submission.to_csv(sub_path, index=False)
print(f"\nSubmission saved → {sub_path}")
submission.head()


## 12. Final Error Analysis

In [ ]:
# OOF from best ensemble (XGB OOF already collected)
best_oof_blend = (opt_weights[0]*oof_xgb + opt_weights[1]*oof_lgb + opt_weights[2]*oof_cat)
best_oof_pred  = apply_thresholds(best_oof_blend, best_thresholds)

print(f"Final ensemble OOF MacroF1 : {f1_score(oof_stk, best_oof_pred, average='macro'):.4f}")
print()
print(classification_report(oof_stk, best_oof_pred, target_names=["Low","Medium","High"]))

cm = confusion_matrix(oof_stk, best_oof_pred)
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(cm, display_labels=["Low","Medium","High"]).plot(
    ax=ax, cmap="Blues", colorbar=True)
ax.set_title("Final Optimised Ensemble — OOF Confusion Matrix")
plt.tight_layout(); plt.show()


In [ ]:
# Confidence distribution per class
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_names = ["Low", "Medium", "High"]
for i, (cls, ax) in enumerate(zip(class_names, axes)):
    mask = oof_stk == i
    conf_correct = best_oof_blend[mask & (best_oof_pred == oof_stk), i]
    conf_wrong   = best_oof_blend[mask & (best_oof_pred != oof_stk), i]
    ax.hist(conf_correct, bins=30, alpha=0.6, color="green", label="Correct", density=True)
    ax.hist(conf_wrong,   bins=30, alpha=0.6, color="red",   label="Wrong",   density=True)
    ax.set_title(f"Class: {cls} — Confidence P(class)")
    ax.set_xlabel("Predicted probability"); ax.legend()
plt.suptitle("Prediction Confidence Distribution", fontsize=13)
plt.tight_layout(); plt.show()
